# _Librerias_

In [13]:
import pandas as pd
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from itertools import product 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_validate, GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import shap
from itertools import product

# _Funciones_

In [14]:
def classificator(df, interval_lst, column_name):
    """Transforms a dataframe's column into classes"""
    array = df[column_name].to_numpy()
    indexes_lst = []
    for i, interval in enumerate(interval_lst):
        lower_limit, upper_limit = interval
        indexes_lst.append(np.intersect1d(np.where(lower_limit < array), np.where(array <= upper_limit)))
    
    classfull = df[column_name].copy()  
    for index, indexes in enumerate(indexes_lst):
        classfull.iloc[indexes] = index

    df_classfull = df.copy()  
    df_classfull[column_name] = classfull  
    
    return df_classfull

In [15]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Greens):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting normalize=True.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)
    fig=plt.figure()
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predict label')

# Carga de datasets

Quindio_Fijo=  pd.read_excel("QuindioFija_1012.xlsx")
Quindio_CropX=  pd.read_excel("QuindioCropX_2612.xlsx")
Quindio_Movil=  pd.read_excel("QuindioMovil_0412.xlsx")
Quindio_Manejo=  pd.read_excel("QuindioManejo_2911.xlsx")
Quindio_Manual=  pd.read_excel("QuindioManual_1112.xlsx")

In [16]:
import os

# Listar los archivos dentro de la carpeta "BD Quindio"
folder_path = R"D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Codigo\BaseDatos\BD Quindio"
archivos = os.listdir(folder_path)
print("Archivos en la carpeta 'BD Quindio':")
print(archivos)

Archivos en la carpeta 'BD Quindio':
['QuindioCropX_2612.xlsx', 'QuindioFija_1012.xlsx', 'QuindioManejo_2911.xlsx', 'QuindioManual_1112.xlsx', 'QuindioMovil_0412.xlsx']


In [17]:
import os
import pandas as pd

folder_path = R"D:\Estudio\OneDrive - Universidad de Antioquia\Estudio\PAI\Codigo\BaseDatos\BD Quindio"
archivos = os.listdir(folder_path)

# Diccionario para guardar los dataframes
dfs = {}

for archivo in archivos:
    nombre_df = archivo.split('.')[0].replace(' ', '_')
    ruta = os.path.join(folder_path, archivo)
    if archivo.endswith('.xlsx'):
        dfs[nombre_df] = pd.read_excel(ruta)
    elif archivo.endswith('.csv'):
        dfs[nombre_df] = pd.read_csv(ruta)

# Asignar los dataframes a variables específicas si existen
Quindio_CropX = dfs.get('QuindioCropX_2612', None)
Quindio_Fijo = dfs.get('QuindioFija_1012', None)
Quindio_Manejo = dfs.get('QuindioManejo_2911', None)
Quindio_Manual = dfs.get('QuindioManual_1112', None)
Quindio_Movil = dfs.get('QuindioMovil_0412', None)

# Exploración de datos 

In [18]:
#Exploración Quindio_fijo
QF_shape= Quindio_Fijo.shape
QF_columnas= Quindio_Fijo.columns
QF_tipos= Quindio_Fijo.dtypes
QF_nulos= Quindio_Fijo.isna().sum()


#Exploración Quindio_CropX
QC_shape= Quindio_CropX.shape
QC_columnas= Quindio_CropX.columns
QC_tipos= Quindio_CropX.dtypes
QC_nulos= Quindio_CropX.isna().sum()


#Exploración Quindio_Movil
QMovil_shape= Quindio_Movil.shape
QMovil_columnas= Quindio_Movil.columns
QMovil_tipos= Quindio_Movil.dtypes
QMovil_nulos= Quindio_Movil.isna().sum()

#Exploración Quindio_Manejo
QManejo_shape= Quindio_Manejo.shape
QManejo_columnas= Quindio_Manejo.columns
QManejo_tipos= Quindio_Manejo.dtypes
QManejo_nulos= Quindio_Manejo.isna().sum()

#Exploración Quindio_Manual
QManual_shape= Quindio_Manual.shape
QManual_columnas= Quindio_Manual.columns
QManual_tipos= Quindio_Manual.dtypes
QManual_nulos= Quindio_Manual.isna().sum()

In [19]:
QC_columnas

Index(['Fecha_Hora', 'VWC-20cm', 'VWC-41cm', 'VWC-66cm', 'Temp(C)-20cm',
       'Temp(C)-41cm', 'EC(dS/m)-20cm', 'EC(dS/m)-41cm', 'EC(dS/m)-66cm',
       'Planta', 'Latitud', 'Longitud', 'Tratamiento'],
      dtype='object')

In [20]:
QManual_columnas

Index(['Fecha', 'Laboratorio', 'Cultivo', 'Planta', 'Posicion_x', 'Posicion_y',
       'Tratamiento', 'Altura planta (cm)', 'Diametro tallo (mm)',
       'Clorofila (SPAD)', 'Numero flores ', 'Numero frutos',
       'Numero frutos cosechados ', 'Peso frutos cosechados (Kg)',
       'Perímetro (mm)', 'Tamanno altura (mm)', 'Frutos de 2da', 'pH_savia',
       'K_savia (ppm)', 'Ca_savia (ppm)', 'Na_savia (ppm)', 'NO3_savia (ppm)',
       'Conductividad_savia (mS/cm)', 'Fosforo_savia (ppm)', 'pH_suelo_Horiba',
       'K_suelo_Horiba (ppm)', 'Ca_suelo_Horiba (ppm)',
       'Na_suelo_Horiba (ppm)', 'NO3_suelo_Horiba (ppm)',
       'Conductividad_suelo_Horiba (mS/cm)', 'Fosforo_suelo_Hanna (ppm)'],
      dtype='object')

In [21]:
QMovil_columnas

Index(['Fecha', 'Laboratorio', 'Cultivo', 'Planta', 'Time',
       '7in1_Nitrogen[mg/kg]', '7in1_Phosphorus[mg/kg]',
       '7in1_Potasium[mg/kg]', '7in1_Ph[pH]', '7in1_Moisture[%RH]',
       '7in1_S_Temperature[C]', '7in1_Conductivity[uS/cm]',
       'Pynamometer_Radiation[W/m^2]', 'Air_sensor_Temperature[C]',
       'Air_sensor_Humidity[%RH]', 'RGB_capture', 'Tapo_capture',
       'Parrot_capture', 'Posicion_x', 'Posicion_y', 'Tratamiento',
       'Tratamiento iniciado'],
      dtype='object')

In [22]:
Quindio_Fijo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 425 entries, 0 to 424
Data columns (total 18 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Fecha                         425 non-null    datetime64[ns]
 1   Laboratorio                   425 non-null    object        
 2   Cultivo                       425 non-null    object        
 3   Planta                        425 non-null    int64         
 4   Time                          425 non-null    datetime64[ns]
 5   7in1_Nitrogen[mg/kg]          234 non-null    float64       
 6   7in1_Phosphorus[mg/kg]        234 non-null    float64       
 7   7in1_Potasium[mg/kg]          234 non-null    float64       
 8   7in1_Ph[pH]                   425 non-null    float64       
 9   7in1_Moisture[%RH]            425 non-null    float64       
 10  7in1_S_Temperature[°C]        425 non-null    float64       
 11  7in1_Conductivity[uS/cm]      42

In [23]:
Quindio_CropX.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20451 entries, 0 to 20450
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Fecha_Hora     20451 non-null  datetime64[ns]
 1   VWC-20cm       20451 non-null  float64       
 2   VWC-41cm       20451 non-null  float64       
 3   VWC-66cm       20451 non-null  float64       
 4   Temp(C)-20cm   20451 non-null  float64       
 5   Temp(C)-41cm   20449 non-null  float64       
 6   EC(dS/m)-20cm  20448 non-null  float64       
 7   EC(dS/m)-41cm  20448 non-null  float64       
 8   EC(dS/m)-66cm  20449 non-null  float64       
 9   Planta         20451 non-null  int64         
 10  Latitud        20451 non-null  float64       
 11  Longitud       20451 non-null  float64       
 12  Tratamiento    20451 non-null  object        
dtypes: datetime64[ns](1), float64(10), int64(1), object(1)
memory usage: 2.0+ MB


In [24]:
Quindio_Movil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   Fecha                         1025 non-null   datetime64[ns]
 1   Laboratorio                   1025 non-null   object        
 2   Cultivo                       1025 non-null   object        
 3   Planta                        1025 non-null   int64         
 4   Time                          1025 non-null   datetime64[ns]
 5   7in1_Nitrogen[mg/kg]          590 non-null    float64       
 6   7in1_Phosphorus[mg/kg]        590 non-null    float64       
 7   7in1_Potasium[mg/kg]          590 non-null    object        
 8   7in1_Ph[pH]                   1025 non-null   float64       
 9   7in1_Moisture[%RH]            1025 non-null   float64       
 10  7in1_S_Temperature[C]         1025 non-null   float64       
 11  7in1_Conductivity[uS/cm]      

In [25]:
Quindio_Manejo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17334 entries, 0 to 17333
Data columns (total 25 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Fecha                        17334 non-null  datetime64[ns]
 1   Planta                       17334 non-null  object        
 2   Tratamiento                  17334 non-null  object        
 3   Posición_x                   17334 non-null  float64       
 4   Posición_y                   17334 non-null  float64       
 5   Evento                       17334 non-null  object        
 6   Detalle                      15714 non-null  object        
 7   Volumen agua por planta (L)  14816 non-null  object        
 8   React1                       13914 non-null  object        
 9   Masa_por_planta1 (g/planta)  5460 non-null   object        
 10  Vol_por_planta1 (L/planta)   13379 non-null  object        
 11  Concent1 (g/L  cc/L)         720 non-null

In [26]:
Quindio_Manual.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2580 entries, 0 to 2579
Data columns (total 31 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   Fecha                               2580 non-null   datetime64[ns]
 1   Laboratorio                         2580 non-null   object        
 2   Cultivo                             2580 non-null   object        
 3   Planta                              2580 non-null   object        
 4   Posicion_x                          2580 non-null   float64       
 5   Posicion_y                          2580 non-null   float64       
 6   Tratamiento                         2580 non-null   object        
 7   Altura planta (cm)                  337 non-null    float64       
 8   Diametro tallo (mm)                 0 non-null      float64       
 9   Clorofila (SPAD)                    593 non-null    float64       
 10  Numero flores           